# Práctica — Clase 3: Construir una red neuronal con PyTorch

Curso **"Deep Learning con Python y PyTorch desde cero"** — DIH DATAlife (Hub de Innovación Digital Europeo), a partir de la certificación EITC/AI/DLPP (European IT Certificate / Artificial Intelligence / Deep Learning with Python and PyTorch).

En la clase de teoría definiste `ClasificadorMNIST`, una red con dos capas ocultas (784 -> 128 -> 64 -> 10) que hereda de `nn.Module`. En esta práctica vas a construir tu propia variante de esa arquitectura y a comprobar que la entiendes de verdad, no solo que sabes copiarla. Hay dos ejercicios:

1. Definir una red con una arquitectura distinta (otro tamaño de capa oculta) y contar cuántos parámetros entrenables tiene
2. Hacerle un forward pass a esa red con un lote de datos aleatorios y comprobar la forma de la salida

Cada ejercicio tiene una celda de código con el andamiaje ya preparado y partes marcadas con `# TODO`. Intenta resolverlas tú antes de abrir la solución, que está oculta justo debajo de cada ejercicio.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Versión de PyTorch:", torch.__version__)

## Ejercicio 1 — Otra arquitectura y su número de parámetros

Define una clase `MiClasificador` que herede de `nn.Module`, con una arquitectura distinta a la de teoría: en vez de dos capas ocultas de 128 y 64 neuronas, usa **una sola capa oculta de 32 neuronas**. La entrada sigue siendo de 784 valores (una imagen MNIST aplanada) y la salida sigue siendo de 10 valores (un dígito del 0 al 9).

Pistas:

- Necesitas dos capas `nn.Linear`: entrada -> capa oculta (`784` a `32`) y capa oculta -> salida (`32` a `10`)
- No olvides llamar a `super().__init__()` al principio de `__init__`
- En el `forward`, aplica `F.relu` tras la capa oculta y `F.log_softmax(..., dim=1)` en la salida, igual que en el ejemplo de teoría
- El número de parámetros entrenables de una capa `nn.Linear(in_features, out_features)` es `in_features * out_features + out_features` (los pesos más un sesgo por neurona). Puedes calcularlo tú a mano o recorrer `modelo.parameters()` sumando `p.numel()` para cada tensor con `p.requires_grad`

In [ ]:
class MiClasificador(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define self.fc1 como una capa nn.Linear de 784 a 32
        self.fc1 = None
        # TODO: define self.fc2 como una capa nn.Linear de 32 a 10
        self.fc2 = None

    def forward(self, x):
        # TODO: aplica F.relu tras self.fc1
        # TODO: aplica F.log_softmax(..., dim=1) tras self.fc2
        # TODO: devuelve el resultado final
        pass


# TODO: instancia tu red en la variable "mi_modelo" y imprímela
mi_modelo = None

# TODO: calcula el número total de parámetros entrenables y guárdalo en "n_parametros"
n_parametros = None
print("Parámetros entrenables:", n_parametros)

<details>
<summary>💡 Ver solución</summary>

```python
class MiClasificador(nn.Module):
    def __init__(self):
        super().__init__()
        # Una sola capa oculta de 32 neuronas, en vez de las dos de 128 y 64 de teoría
        self.fc1 = nn.Linear(784, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.log_softmax(self.fc2(x), dim=1)
        return x


mi_modelo = MiClasificador()
print(mi_modelo)

# Sumamos el número de elementos de cada tensor de parámetros (pesos y sesgos)
n_parametros = sum(p.numel() for p in mi_modelo.parameters() if p.requires_grad)
print("Parámetros entrenables:", n_parametros)

# Comprobación manual:
# fc1: 784 * 32 pesos + 32 sesgos = 25 088 + 32 = 25 120
# fc2: 32 * 10 pesos + 10 sesgos = 320 + 10 = 330
# Total = 25 450
```

</details>

## Ejercicio 2 — Forward pass con datos aleatorios

Ahora vas a comprobar que tu red del ejercicio 1 procesa correctamente un lote de datos, sin necesidad de cargar el dataset MNIST real. Para eso vas a simular un lote con `torch.randn`, que genera valores aleatorios con la misma forma que un lote real ya aplanado.

Pistas:

- Un lote de `n` imágenes MNIST aplanadas tiene forma `(n, 784)`
- Usa `torch.randn(n, 784)` para crear un lote simulado (elige tú el tamaño de lote, por ejemplo 8 o 32)
- Pasa ese lote por `mi_modelo` (el de tu ejercicio 1) y guarda el resultado
- La salida debería tener forma `(n, 10)`: una fila por imagen del lote, diez valores por fila
- Comprueba también que, tras aplicar `torch.exp` a una fila de la salida, sus valores suman aproximadamente 1 (porque `log_softmax` produce una distribución de probabilidad en escala logarítmica)

In [ ]:
# TODO: crea un lote simulado de 8 imágenes aplanadas con torch.randn
lote_aleatorio = None

# TODO: pasa el lote por mi_modelo (el del ejercicio 1) y guarda el resultado en "salida"
salida = None

# TODO: imprime la forma de "salida" (debería ser (8, 10))

# TODO: aplica torch.exp a la primera fila de "salida" y comprueba que sus valores suman
#       aproximadamente 1


<details>
<summary>💡 Ver solución</summary>

```python
# Lote simulado de 8 imágenes MNIST ya aplanadas (8 x 784)
lote_aleatorio = torch.randn(8, 784)

# Forward pass a través de la red del ejercicio 1
salida = mi_modelo(lote_aleatorio)

print("Forma de la salida:", salida.shape)  # torch.Size([8, 10])

# torch.exp deshace el logaritmo de log_softmax y recupera probabilidades normales
probabilidades = torch.exp(salida[0])
print("Probabilidades de la primera imagen:", probabilidades)
print("Suma de las probabilidades:", probabilidades.sum().item())  # aproximadamente 1.0
```

</details>

## Para seguir practicando

Si te ha sobrado tiempo, prueba a añadir una segunda capa oculta a `MiClasificador` (por ejemplo, 784 -> 32 -> 16 -> 10) y repite los dos ejercicios. Comprueba cómo cambia el número de parámetros entrenables al añadir esa capa: es la mejor forma de comprobar que entiendes de dónde sale cada parámetro, y no solo que sabes repetir la fórmula.